# MarketPulse: сегментація клієнтів e-commerce
**Мета:** науково обґрунтувати кількість сегментів (Elbow Method + Silhouette) та порівняти **K-Means** з **Agglomerative Clustering**.

План:
1. Дані та підготовка (RFM-подібні ознаки)
2. Метод ліктя + silhouette для вибору k
3. K-Means
4. Ієрархічна кластеризація (дендрограма + Agglomerative)
5. Порівняння метрик і візуалізація
6. Профілі сегментів і висновки

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import (silhouette_score, davies_bouldin_score,
                             calinski_harabasz_score, adjusted_rand_score)
from scipy.cluster.hierarchy import linkage, dendrogram

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.rcParams["figure.figsize"] = (8, 5)

## 1. Дані
Реальної вибірки немає, тому генеруємо синтетичну: 4 «прихованих» типи клієнтів.
**Щоб використати свої дані**, замініть клітинку нижче на `df = pd.read_csv("your_file.csv")` і вкажіть список `features`.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)

# (n, mean, std) для кожної ознаки
profiles = {
    "VIP":          dict(n=300, recency=(10, 5),   frequency=(40, 8),  monetary=(2500, 500), avg_basket=(90, 20),  discount_share=(0.10, 0.05)),
    "Лояльні":      dict(n=600, recency=(25, 10),  frequency=(18, 5),  monetary=(900, 200),  avg_basket=(50, 12),  discount_share=(0.25, 0.08)),
    "Мисливці за знижками": dict(n=500, recency=(45, 15), frequency=(10, 4), monetary=(350, 100), avg_basket=(35, 10), discount_share=(0.70, 0.10)),
    "Сплячі":       dict(n=600, recency=(180, 40), frequency=(3, 1.5), monetary=(120, 50),   avg_basket=(40, 15),  discount_share=(0.30, 0.12)),
}

frames = []
for name, p in profiles.items():
    n = p["n"]
    d = {col: rng.normal(*p[col], n) for col in
         ["recency", "frequency", "monetary", "avg_basket", "discount_share"]}
    d["true_segment"] = name
    frames.append(pd.DataFrame(d))

df = pd.concat(frames, ignore_index=True)
df["recency"] = df["recency"].clip(lower=1)
df["frequency"] = df["frequency"].clip(lower=1).round()
df["monetary"] = df["monetary"].clip(lower=10)
df["avg_basket"] = df["avg_basket"].clip(lower=5)
df["discount_share"] = df["discount_share"].clip(0, 1)
df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

features = ["recency", "frequency", "monetary", "avg_basket", "discount_share"]
df.head()

In [ ]:
print(df.shape)
display(df[features].describe().T)
print("Пропуски:", df[features].isna().sum().sum())

In [ ]:
# Масштабування обов'язкове: K-Means і Ward працюють з евклідовою відстанню
X = StandardScaler().fit_transform(df[features])
X.shape

## 2. Вибір кількості кластерів
**Elbow Method:** будуємо inertia (WCSS) для k = 1…10. «Лікоть» — точка, після якої приріст якості різко сповільнюється.
Додатково перевіряємо **Silhouette**, бо лікоть інколи неоднозначний.

In [ ]:
K_RANGE = range(1, 11)
inertias, silhouettes = [], {}

for k in K_RANGE:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X)
    inertias.append(km.inertia_)
    if k >= 2:
        silhouettes[k] = silhouette_score(X, km.labels_)

def find_elbow(ks, values):
    # Точка з максимальною відстанню до прямої, що з'єднує першу й останню точки кривої
    ks = np.array(list(ks), dtype=float)
    v = np.array(values, dtype=float)
    p = np.column_stack([(ks - ks.min()) / (ks.max() - ks.min()),
                         (v - v.min()) / (v.max() - v.min())])
    a, b = p[0], p[-1]
    line = (b - a) / np.linalg.norm(b - a)
    vecs = p - a
    dist = np.abs(vecs[:, 0] * line[1] - vecs[:, 1] * line[0])
    return int(ks[np.argmax(dist)])

k_elbow = find_elbow(K_RANGE, inertias)
k_sil = max(silhouettes, key=silhouettes.get)
print(f"Лікоть: k = {k_elbow}")
print(f"Найкращий silhouette: k = {k_sil} ({silhouettes[k_sil]:.3f})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(list(K_RANGE), inertias, "o-")
axes[0].axvline(k_elbow, color="red", ls="--", label=f"лікоть k={k_elbow}")
axes[0].set(title="Elbow Method (inertia / WCSS)", xlabel="k", ylabel="Inertia")
axes[0].legend()

axes[1].plot(list(silhouettes.keys()), list(silhouettes.values()), "o-", color="green")
axes[1].axvline(k_sil, color="red", ls="--", label=f"max silhouette k={k_sil}")
axes[1].set(title="Silhouette score", xlabel="k", ylabel="Silhouette")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Підсумкове значення k (лікоть як основний критерій, silhouette як перевірка)
K = k_elbow
print("Обране K =", K)

## 3. K-Means

In [ ]:
kmeans = KMeans(n_clusters=K, n_init=20, random_state=RANDOM_STATE)
df["kmeans"] = kmeans.fit_predict(X)
df["kmeans"].value_counts().sort_index()

## 4. Ієрархічна кластеризація
Спочатку дендрограма (на підвибірці — на повних даних вона надто важка), потім `AgglomerativeClustering`.

In [ ]:
sample_idx = rng.choice(len(X), size=400, replace=False)
Z = linkage(X[sample_idx], method="ward")

plt.figure(figsize=(12, 5))
dendrogram(Z, truncate_mode="lastp", p=30, show_leaf_counts=True, leaf_rotation=90)
plt.title("Дендрограма (Ward, підвибірка 400 клієнтів)")
plt.xlabel("Кластери / кількість клієнтів")
plt.ylabel("Відстань злиття")
plt.show()

In [ ]:
# Порівняємо різні linkage; метрика euclidean
linkage_results = {}
for link in ["ward", "complete", "average", "single"]:
    labels = AgglomerativeClustering(n_clusters=K, linkage=link).fit_predict(X)
    linkage_results[link] = silhouette_score(X, labels)

pd.Series(linkage_results, name="silhouette").sort_values(ascending=False).to_frame()

In [ ]:
agg = AgglomerativeClustering(n_clusters=K, linkage="ward")
df["agglo"] = agg.fit_predict(X)
df["agglo"].value_counts().sort_index()

## 5. Порівняння K-Means та Agglomerative

In [ ]:
def evaluate(labels):
    return {
        "Silhouette (↑)": silhouette_score(X, labels),
        "Davies-Bouldin (↓)": davies_bouldin_score(X, labels),
        "Calinski-Harabasz (↑)": calinski_harabasz_score(X, labels),
    }

comparison = pd.DataFrame({
    "K-Means": evaluate(df["kmeans"]),
    "Agglomerative (Ward)": evaluate(df["agglo"]),
}).T

# Узгодженість двох методів між собою та з «істинними» сегментами (доступні лише на синтетиці)
comparison["ARI vs true"] = [
    adjusted_rand_score(df["true_segment"], df["kmeans"]),
    adjusted_rand_score(df["true_segment"], df["agglo"]),
]
print("ARI між K-Means та Agglomerative:",
      round(adjusted_rand_score(df["kmeans"], df["agglo"]), 3))
comparison.round(3)

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X2 = pca.fit_transform(X)
print("Пояснена дисперсія PC1+PC2:", pca.explained_variance_ratio_.sum().round(3))

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)
for ax, col, title in zip(axes, ["kmeans", "agglo"], ["K-Means", "Agglomerative (Ward)"]):
    sc = ax.scatter(X2[:, 0], X2[:, 1], c=df[col], cmap="tab10", s=12, alpha=0.7)
    ax.set(title=title, xlabel="PC1", ylabel="PC2")
    ax.legend(*sc.legend_elements(), title="Кластер")
plt.tight_layout()
plt.show()

In [ ]:
# Як кластери одного методу розподіляються по кластерах іншого
pd.crosstab(df["kmeans"], df["agglo"], rownames=["K-Means"], colnames=["Agglomerative"])

## 6. Профілі сегментів (для маркетингу)

In [ ]:
profile = df.groupby("kmeans")[features].mean().round(2)
profile["size"] = df["kmeans"].value_counts().sort_index()
profile["share_%"] = (profile["size"] / len(df) * 100).round(1)
profile

In [ ]:
# Нормалізований профіль для теплової карти
z = (profile[features] - profile[features].mean()) / profile[features].std()

plt.figure(figsize=(8, 4))
plt.imshow(z, cmap="RdYlGn", aspect="auto")
plt.colorbar(label="z-score відносно інших кластерів")
plt.xticks(range(len(features)), features, rotation=30)
plt.yticks(range(len(z)), [f"Кластер {i}" for i in z.index])
for i in range(z.shape[0]):
    for j in range(z.shape[1]):
        plt.text(j, i, f"{z.iloc[i, j]:.1f}", ha="center", va="center")
plt.title("Профілі кластерів K-Means")
plt.tight_layout()
plt.show()

## Висновки
- **Кількість сегментів.** Метод ліктя вказує на оптимальне `K` (див. графік); його підтверджує максимум silhouette. Додавання кластерів понад це значення дає незначне зниження inertia і ускладнює інтерпретацію для маркетингу.
- **K-Means vs Agglomerative.** Порівняйте таблицю метрик: Silhouette / Calinski-Harabasz (чим більше, тим краще) та Davies-Bouldin (чим менше, тим краще). Високий ARI між методами означає, що структура даних стійка і не залежить від алгоритму.
- **Практика.** K-Means швидший і масштабується на мільйони клієнтів; Agglomerative дає дендрограму та корисний для виявлення вкладених сегментів, але має складність O(n²) за пам'яттю, тож для великих вибірок застосовується на підвибірці або на центроїдах.
- **Далі:** дати кластерам бізнес-назви за профілями та розробити окремі кампанії (утримання VIP, реактивація «сплячих», обмеження залежності від знижок).